# E15 时间序列：分解、平稳性与基线预测

> 覆盖矩阵：时序（相对 E08 更偏「预测/分解」）

## 你将学到
1. 经典季节分解（趋势 / 季节 / 残差）。
2. ADF 平稳性提示；差分的直觉。
3. 朴素 / 季节朴素 / 线性趋势基线；**禁止随机打乱评估**。
4. 与 `extract_datetime_features`、`detect_time_leakage` 衔接。
5. **缺口**：无 `TimeSeriesTools.decompose / forecast_baseline`。

> **本课增补：** 月度同比边缘情况（缺月 / 零基数 / 未完月）。


In [ ]:
QUICK = True
RANDOM_STATE = 42
from wanglaoshi.runtime import ready
ready(seed=RANDOM_STATE)


## 1. 合成日序列（趋势+周季节+噪声）

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

rng = np.random.default_rng(RANDOM_STATE)
n = 365 * 2
dates = pd.date_range('2022-01-01', periods=n, freq='D')
t = np.arange(n)
trend = 100 + 0.05 * t
season = 12 * np.sin(2 * np.pi * t / 7)
noise = rng.normal(0, 5, n)
y = trend + season + noise
df = pd.DataFrame({'ds': dates, 'y': y})
print(df.head(), df.shape)


## 2. 季节分解（statsmodels）

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose
import matplotlib.pyplot as plt

s = df.set_index('ds')['y']
decomp = seasonal_decompose(s, model='additive', period=7)
fig = decomp.plot()
fig.set_size_inches(8, 6)
plt.tight_layout()
out = Path('outputs/15_timeseries_forecast'); out.mkdir(parents=True, exist_ok=True)
fig.savefig(out / 'decompose.png', dpi=100)
plt.close()
print('saved decompose.png')


## 3. ADF 与差分

In [ ]:
from statsmodels.tsa.stattools import adfuller

def adf_report(series, name):
    r = adfuller(series.dropna(), autolag='AIC')
    print(name, 'adf=', round(r[0], 3), 'p=', round(r[1], 4))

adf_report(s, 'level')
adf_report(s.diff(), 'diff1')


## 4. wanglaoshi 时间特征 + 泄漏意识

In [ ]:
from wanglaoshi import features as FE
from wanglaoshi import compete as CT

fe_df = df.rename(columns={'ds': 'datetime'}).copy()
eng = FE.AdvancedFeatureEngineer()
fe_df = eng.extract_datetime_features(fe_df, 'datetime')
print([c for c in fe_df.columns if c not in ('datetime', 'y')][:12])
# 故意制造未来泄漏列做对照
fe_df['y_lead1'] = fe_df['y'].shift(-1)
leak = CT.LeakageDetector()
# 接口：列名启发式
try:
    warnings = leak.detect_time_leakage(fe_df, date_col='datetime', target_col='y')
    print('leakage warnings:', warnings)
except Exception as e:
    print('detect_time_leakage note:', type(e).__name__, e)


## 5. 时间切分基线预测

In [ ]:
from sklearn.metrics import mean_absolute_error

split = int(len(df) * 0.8)
train, test = df.iloc[:split], df.iloc[split:]
# 季节朴素：上周同星期
pred_season = test['y'].copy()
hist = train['y'].to_numpy()
for i, idx in enumerate(test.index):
    pred_season.iloc[i] = hist[-(7 - (i % 7))] if len(hist) >= 7 else hist[-1]
    # 更稳：用训练末尾对齐星期
# 简化：用 shift(7) 在全序列上再切片
df['naive7'] = df['y'].shift(7)
pred = df.iloc[split:]['naive7']
mae = mean_absolute_error(test['y'], pred)
print('seasonal naive MAE', round(mae, 3))
# 线性趋势（仅用时间索引）
from sklearn.linear_model import LinearRegression
X_tr = np.arange(split).reshape(-1, 1)
X_te = np.arange(split, len(df)).reshape(-1, 1)
lr = LinearRegression().fit(X_tr, train['y'])
mae_lr = mean_absolute_error(test['y'], lr.predict(X_te))
print('linear trend MAE', round(mae_lr, 3))
pd.DataFrame({'mae_naive7': [mae], 'mae_linear': [mae_lr]}).to_csv(out / 'forecast_metrics.csv', index=False)


## 6. 月度销售同比：缺月、零基数与未完月

> 对照：[DAX SAMEPERIODLASTYEAR](https://learn.microsoft.com/en-us/dax/sameperiodlastyear-function-dax)

教学点：对齐「去年同月」前必须先 **补齐日历骨架**；区分缺月（无记录）与真实 0；零基数导致 YoY 无穷大；未完月不可与去年整月比。


In [ ]:
# 小销售表：故意缺 2023-02，2023-03 为真实 0，2024-11 为未完月
sales_m = pd.DataFrame({
    'month': pd.to_datetime([
        '2023-01-01', '2023-03-01', '2023-04-01',
        '2024-01-01', '2024-02-01', '2024-03-01', '2024-04-01', '2024-11-01',
    ]),
    'sales': [100.0, 0.0, 120.0, 110.0, 130.0, 0.0, 140.0, 50.0],
})
# 完整月骨架
idx = pd.date_range('2023-01-01', '2024-11-01', freq='MS')
cal = pd.DataFrame({'month': idx}).merge(sales_m, on='month', how='left')
cal['is_missing'] = cal['sales'].isna()  # 缺月
cal['sales_filled'] = cal['sales']  # 缺月保持 NaN，真实 0 保留 0
cal['sales_for_sum'] = cal['sales'].fillna(0)  # 仅用于某些展示——同比时不要盲目 fillna(0) 当「真零」

def yoy_row(row, df):
    prev = row['month'] - pd.DateOffset(years=1)
    hit = df.loc[df['month'] == prev]
    if hit.empty:
        return dict(yoy=np.nan, reason='no_prior_period')
    prev_sales = hit['sales'].iloc[0]
    cur = row['sales']
    if pd.isna(cur):
        return dict(yoy=np.nan, reason='current_missing_month')
    if pd.isna(prev_sales):
        return dict(yoy=np.nan, reason='prior_missing_month')
    if prev_sales == 0:
        return dict(yoy=np.nan, reason='zero_base')
    return dict(yoy=(cur - prev_sales) / prev_sales, reason='ok')

# 未完月：假设「今天」是 2024-11-15，则 2024-11 不可比去年 11 整月
as_of = pd.Timestamp('2024-11-15')
cal['incomplete'] = (cal['month'].dt.to_period('M') == as_of.to_period('M')) & (as_of.day < cal['month'].dt.days_in_month)

rows = []
for _, r in cal.iterrows():
    y = yoy_row(r, cal)
    if r['incomplete']:
        y['reason'] = 'incomplete_month'
        y['yoy'] = np.nan
    rows.append({**r.to_dict(), **y})
yoy_df = pd.DataFrame(rows)
print(yoy_df[['month', 'sales', 'is_missing', 'incomplete', 'yoy', 'reason']].to_string(index=False))
out = Path('outputs/15_timeseries_forecast'); out.mkdir(parents=True, exist_ok=True)
yoy_df.to_csv(out / 'yoy_edge_cases.csv', index=False)
print('saved yoy_edge_cases.csv')


## 附录
- 作业：换 `period=30` 看月季节是否过度；对比 E08 Bike 小时级。
- 包优化：`TimeSeriesTools` 一键分解图 + ADF 表 + 安全滞后工厂。
